### K-Nearest Neighbor Algorithm 
- KNN is a **non-parametric**, **instance-based**, **lazy learning** algorithm (no training phase; prediction happens at inference).
- It is a **local method**: prediction for a point depends only on its nearest neighbors.
- Given a query point x:
    - compute distance from x to every training point
    - select the k closest points (nearest neighbors)
- Prediction rules:
    - Regression: take the average of the target values of the k nearest neighbors
    - Classification: assign the class that appears most frequently among the k nearest neighbors

### Properties
- Feature scaling is mandatory; otherwise distance is dominated by large-scale features.
- Assumes **local smoothness**: nearby points have similar outputs.
- Suffers from **curse of dimensionality**: distances become less meaningful in high dimensions.
- Sensitive to irrelevant/correlated features; preprocessing is crucial.

In [36]:
import numpy as np

np.random.seed(42)

# number of samples per class
n0 = 100
n1 = 100

# Class 0: cluster around (2,2)
X0 = np.random.normal(loc=2.0, scale=0.8, size=(n0, 2))

# Class 1: cluster around (6,6)
X1 = np.random.normal(loc=6.0, scale=0.8, size=(n1, 2))

# Add overlap (boundary noise)
X0[:10] += np.array([2.5, 2.5])  # push some class 0 points toward center
X1[:10] -= np.array([2.5, 2.5])  # push some class 1 points toward center

# Combine
X_train = np.vstack([X0, X1])
y_train = np.hstack([np.zeros(n0), np.ones(n1)])

# Add outliers (important for KNN behavior)
outliers = np.array([
    [6.5, 2.0],  # class 0 region but label 1
    [2.0, 6.5],  # class 1 region but label 0
])

X_train = np.vstack([X_train, outliers])
y_train = np.hstack([y_train, [1, 0]])

# Query point near boundary
x_query = np.array([4.5, 4.5])

In [37]:
X_train.shape

(202, 2)

In [38]:
y_train.shape

(202,)

In [39]:
X_train[:5,:]

array([[4.89737132, 4.38938856],
       [5.01815083, 5.71842389],
       [4.3126773 , 4.31269043],
       [5.76337025, 5.11394778],
       [4.12442049, 4.93404803]])

In [40]:
y_train[:5]

array([0., 0., 0., 0., 0.])

In [41]:
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

In [42]:
distances =np.array([euclidean_distance(x_query,i) for i in X_train])

In [43]:
distances.shape

(202,)

In [44]:
distances

array([0.41247892, 1.32402305, 0.26490502, 1.4046481 , 0.57398403,
       0.5256068 , 1.54281551, 1.45140126, 0.84836908, 1.34321631,
       2.9913099 , 4.38530946, 3.7988657 , 4.06686375, 4.04409138,
       3.1504324 , 4.18341377, 3.93447584, 4.68923682, 4.263697  ,
       3.03783945, 3.77274333, 4.79834916, 3.3113549 , 4.49917474,
       3.59250849, 3.64605272, 2.42615679, 4.19590926, 2.81994272,
       3.91514269, 4.83831523, 2.32909234, 3.06948818, 3.73951727,
       2.54947363, 2.81998958, 4.95137019, 3.66195439, 4.75572252,
       3.47314551, 3.1986373 , 4.28024213, 2.8511141 , 3.59363162,
       2.97382515, 4.12347126, 4.62511538, 3.22040891, 3.66785105,
       4.60862268, 4.19130916, 3.41322023, 2.55995643, 3.43708913,
       4.75792679, 2.50832054, 3.48498665, 4.26479164, 2.47360119,
       3.72869427, 3.87470756, 2.16378078, 4.42272808, 3.7794027 ,
       4.46881717, 3.96492417, 3.47223953, 4.16896548, 3.9449416 ,
       2.73646118, 4.45723865, 2.96101341, 4.98244589, 3.07491

In [45]:
k = 3
neighbors = np.argsort(distances)[:k]
neighbors

array([101,   2,   0])

In [46]:
X_train[neighbors]

array([[4.36644099, 4.34304164],
       [4.3126773 , 4.31269043],
       [4.89737132, 4.38938856]])

### Weighted KNN & Tie Handling

**Weighted KNN:** : $w_i = \frac{1}{d_i + \epsilon}, \quad \text{score}(c)=\sum w_i$ . Closer points have higher influence

**Tie-break (unweighted):**
- find classes with max votes  
- scan neighbors by distance (closest → farthest)  
- first match among tied classes is selected  

→ closest tied-class neighbor decides output

In [47]:
import numpy as np

def knn_predict(X_train, y_train, x_query, k=3, weighted=False):

    # Compute Euclidean distances
    distances = np.sqrt(np.sum((X_train - x_query) ** 2, axis=1))

    # Get k nearest neighbors
    nearest_idx = np.argsort(distances)[:k]
    nearest_labels = y_train[nearest_idx]
    nearest_distances = distances[nearest_idx]

    # Unique classes in neighborhood
    classes = np.unique(nearest_labels)

    scores = []

    for c in classes:

        mask = nearest_labels == c

        if weighted:
            # inverse distance weighting : we need a weight function that decreases with distance
            weights = 1 / (nearest_distances[mask] + 1e-12)
            score = np.sum(weights)
        else:
            # simple vote
            score = np.sum(mask)

        scores.append(score)

    scores = np.array(scores)

    # handle ties explicitly: choose class of closest neighbor among tied
    best_classes = classes[np.where(scores == np.max(scores))[0]]

    if len(best_classes) == 1:
        predicted_class = best_classes[0]
    else:
        # tie-breaker: closest neighbor among tied classes
        for idx in nearest_idx:
            if y_train[idx] in best_classes:
                predicted_class = y_train[idx]
                break

    return predicted_class, nearest_idx

In [48]:
knn_predict(X_train,y_train,x_query)

(np.float64(0.0), array([101,   2,   0]))

### K-Nearest Neighbor Regression

In [49]:
import numpy as np

def knn_regression(X_train, y_train, x_query, k=3, weighted=False):

    # compute Euclidean distances
    distances = np.sqrt(np.sum((X_train - x_query) ** 2, axis=1))

    # get k nearest neighbors
    nearest_idx = np.argsort(distances)[:k]
    nearest_distances = distances[nearest_idx]
    nearest_values = y_train[nearest_idx]

    if weighted:
        # inverse distance weighting (prevents dominance by distant points)
        weights = 1 / (nearest_distances + 1e-12)
        prediction = np.sum(weights * nearest_values) / np.sum(weights)
    else:
        # simple average
        prediction = np.mean(nearest_values)

    return prediction, nearest_idx

In [50]:
# Min-Max Scaling (rescale to range [0, 1]):
# X_scaled = (X - X_min) / (X_max - X_min)

# Standard Scaling / Z-score normalization:
# X_scaled = (X - mean(X)) / std(X) ( mean=0 and sd =1)

### Distance Metrics
- Euclidean distance: $d(x,z)=\|x-z\|_2$ (assumes same scale, uncorrelated features)
- Mahalanobis distance (accounts for covariance):$d_M(x,z)=\sqrt{(x-z)^T \Sigma^{-1} (x-z)}$
- If data is whitened ($\Sigma = I$), Mahalanobis reduces to Euclidean.


In [51]:

# Covariance matrix of training data
cov_matrix = np.cov(X_train.T)
cov_inv = np.linalg.inv(cov_matrix)

def mahalanobis_distance(x, data, cov_inv):
    diffs = data - x
    return np.sqrt(np.diag(diffs @ cov_inv @ diffs.T))

maha_distances = mahalanobis_distance(x_query, X_train, cov_inv)
maha_distances


array([0.42935019, 0.7731256 , 0.09444959, 0.68819455, 0.68816043,
       0.18757006, 1.53372955, 0.87173288, 0.89671686, 0.60742066,
       1.60499791, 1.92309708, 1.37529443, 1.66682972, 1.43422901,
       1.85535512, 1.70215576, 2.02524937, 2.29262496, 1.72328465,
       1.18038066, 1.36332991, 1.72857317, 1.4555587 , 2.21124045,
       1.4062726 , 1.46880655, 0.87435238, 1.49852362, 1.04660272,
       1.38838852, 1.73231751, 0.86669007, 1.233672  , 1.54530821,
       1.1237973 , 1.36845642, 2.70283313, 1.35552967, 2.27275547,
       1.25450455, 1.80810227, 1.51762317, 1.12267126, 1.38349687,
       1.14703588, 1.46409937, 1.85616083, 1.15089093, 1.33343213,
       1.6996167 , 1.55481514, 1.23267299, 1.50456606, 1.26677651,
       1.98575799, 1.72332521, 1.24936065, 1.75768197, 0.94331565,
       1.81246242, 2.38772586, 1.23169185, 1.57127338, 1.4426945 ,
       1.81275287, 1.64082507, 1.92347671, 1.48431848, 2.02857158,
       1.13912051, 1.86699791, 1.07346743, 1.78310667, 1.12142

In [52]:
import numpy as np

def knn_mahalanobis(X_train, y_train, x_query, k=3):

    # covariance matrix of training data
    cov_matrix = np.cov(X_train.T)

    # regularization for numerical stability (important in practice)
    cov_matrix += 1e-6 * np.eye(cov_matrix.shape[0])

    cov_inv = np.linalg.inv(cov_matrix)

    # Mahalanobis distances
    diff = X_train - x_query
    distances = np.sqrt(np.sum(diff @ cov_inv * diff, axis=1))

    # nearest neighbors
    nearest_idx = np.argsort(distances)[:k]
    nearest_labels = y_train[nearest_idx]

    # majority voting (robust to arbitrary labels)
    classes = np.unique(nearest_labels)
    scores = np.array([np.sum(nearest_labels == c) for c in classes])

    # handle ties deterministically (choose class of closest neighbor among ties)
    best_classes = classes[np.where(scores == np.max(scores))[0]]

    if len(best_classes) == 1:
        predicted_class = best_classes[0]
    else:
        for idx in nearest_idx:
            if y_train[idx] in best_classes:
                predicted_class = y_train[idx]
                break

    return predicted_class, nearest_idx

pred_class_maha, nearest_idx_maha = knn_mahalanobis(X_train, y_train, x_query, k=3)
pred_class_maha, nearest_idx_maha


(np.float64(1.0), array([101,   2, 173]))

### Choosing k 
- Small $k$: low bias, high variance (noise-sensitive)
- Large $k$: high bias, low variance (over-smoothing)
 - The curve usually decreases error sharply at first (as k increases from 1). Then it flattens.
 - Smallest k within the flat region that gives near-minimum error


In [57]:
import numpy as np
import matplotlib.pyplot as plt

def plot_knn_elbow(X_train, y_train, X_val, y_val, k_range=range(1, 21), weighted=False):
    errors = []

    for k in k_range:
        # for each k we make the predictions for the entire validation dataset 
        preds = []

        for x in X_val:
            pred, _ = knn_predict(X_train, y_train, x, k=k, weighted=weighted)
            preds.append(pred)

        preds = np.array(preds)

        # classification error
        error = np.mean(preds != y_val)
        errors.append(error)
    # for each k we have corresponding errors 
    errors = np.array(errors)

    plt.figure(figsize=(8, 5))
    plt.plot(list(k_range), errors, marker='o')
    plt.xlabel("k (number of neighbors)")
    plt.ylabel("Validation Error")
    plt.title("KNN Elbow Curve")
    plt.grid(True)
    plt.show()

    return k_range, errors